In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from torch.utils.data import DataLoader
import torch.utils.data as data
import torch.nn as nn
from sklearn.preprocessing import RobustScaler
import sys
from itertools import product

!rm -r /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git clone https://github.com/KirillVishnyakov/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git -C /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection log --oneline -1

rm: cannot remove '/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection': No such file or directory
Cloning into 'Architectural-Biases-in-Time-Series-Anomaly-Detection'...
remote: Enumerating objects: 228, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 228 (delta 7), reused 17 (delta 5), pack-reused 207 (from 1)
Receiving objects: 100% (228/228), 2.82 MiB | 35.60 MiB/s, done.
Resolving deltas: 100% (118/118), done.
3fcd236 (HEAD -> main, origin/main, origin/HEAD) xavier fix for linear layer, and saving initial weights for training loop


In [2]:
sys.path.append("/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection")
from dataset import LSTM_Dataset
from lstm import LSTMBaseline
import train
if torch.cuda.is_available():
    device = torch.device("cuda")

### Set random states for reproducibility

In [ ]:
seed = 432
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### Create train and test datasets

In [3]:
train_dataset = LSTM_Dataset(device, 100, 10)
test_dataset = LSTM_Dataset(device, 100, 10, train_dataset.scaler, start = 90000, end = 100000, train = False)

### Create hyperparam grid

In [25]:
hyperparam_grid = {
    'hidden_size': [64, 128],
    'l': [4, 12],
    'num_layers': [1, 2],
    'lr': [0.001, 0.0005, 0.0001],
    'batch_size': [64]
}
experiments = {}
for v in product(*hyperparam_grid.values()):
    experiments[f"hs{v[0]}_l{v[1]}_ly{v[2]}_lr{v[3]}_bs{v[4]}"] = v

### hyperparam sweep

In [ ]:
experiment_results = {}
for experiment_name, h_params in experiments.items():
    model = LSTMBaseline(h_params[0], h_params[1], h_params[2]).to(device)
    train_dataset.l, test_dataset.l = h_params[1], h_params[1]
    experiment_results[experiment_name] = \
    train.fit_lstm(device, model, experiment_name, train_dataset, test_dataset, h_params[3], h_params[4], 2)